# Agent Failure Modes + Debugging

## Problem Statement
Stress-test DevContext AI against 5 adversarial inputs, classify failure modes and implement guards for at least 2.

## My Approach

Going in, I was most worried about A1 - the missing-function case. My gut said the LLM would fill in the gap with something plausible-sounding, and that's the scariest kind of failure because there's no obvious signal it happened. I decided to lean on the `qa_retrieve_generate` prompt to be the primary guard: instruct the model to answer *only from context*, and trust the system prompt to hold that line.

For the parallel search architecture, I knew that returning `{"sources": {...}}` from three nodes simultaneously would cause last-write-wins collisions on the state dict. My fix was a `merge_sources` reducer registered as a LangGraph `Annotated` reducer on the `sources` field - so each parallel node's dict gets merged by category key rather than overwritten. `sources_cited` exists as a separate list specifically so the final answer node has a clean, non-overridable record of what was actually cited, independent of the raw retrieval sources dict.

For A3 (ambiguous query), I knew the right answer was a clarifying interrupt - but that requires branching before routing, which would mean a second classification step. I accepted the trade-off and let the agent pick a branch (it picks QA, which is the safer default), flagging this as a known gap to address later.

## What I Learned

The biggest lesson was that **LangGraph parallel fan-out and shared state fields don't mix without reducers**. Without `merge_sources`, whichever search node wrote last would silently clobber the others - no error, just missing context. The `Annotated[dict, merge_sources]` pattern is the right tool for this, and it's worth understanding it deeply because any production multi-source RAG pipeline hits this problem.

The second lesson: **the safest guard is a tight prompt constraint, not a code guard**. For A1, the model correctly said "no information in context" because the prompt explicitly told it to answer only from provided context. The `NO_RESULTS` sentinel pattern is still worth adding (it gives a deterministic signal to log), but the LLM's grounding instruction is what actually prevented hallucination in the output.

I also learned that `diff_parser` using a regex on `+def` / `-def` is a reliable zero-hallucination approach for the config-only diff case (A2) - no LLM call needed. Parse first, reason second.

## Where I Got Stuck

**Parallel source overwriting** was the main blocker. When `search_codebase`, `search_docs`, and `search_schema` all run in parallel and each returns `{"sources": {"codebase": [...]}}`, LangGraph's default state update is last-write-wins. So for a query that fans out to all three, only one category would survive into `aggregate_search_res`.

The fix was `merge_sources` - a plain dict reducer that does a category-level merge instead of a full dict replacement. Registering it as `Annotated[dict, merge_sources]` on the `sources` field in `AgentState` tells LangGraph to call it instead of overwriting. Once that clicked, parallel retrieval worked correctly.

`sources_cited` is a companion to this: it's a flat list on state, only written by `qa_retrieve_generate`, specifically so there's a clean final-answer citation trail that doesn't get touched by the parallel nodes.

## What I'd Do Differently

**A3 (ambiguous query) is unguarded** - the agent silently picks QA and answers about auth without telling the user it made a choice. The right fix is a pre-routing ambiguity detector: if query token count is below a threshold (say, ≤ 3 words) *and* the intent classifier's response doesn't contain clear mode signal, fire a HITL interrupt before routing. That would handle one-word queries cleanly without touching the main QA/impact branches.

**The loop detector for A5 is implicit, not explicit.** Because LangGraph's graph is directed and each search node fires exactly once per invocation, the loop case doesn't trigger - but that's architecture preventing it, not a guard. A real production system could still loop if I introduced a ReAct-style agent node. I'd add an explicit `tool_call_counter` to state and a guard in `llm_parallel_router_node` that caps re-routing at 1 attempt.

In [1]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

from typing import TypedDict
from langgraph.graph import StateGraph, START , END
from langchain_groq import ChatGroq
import re
from typing import TypedDict, Annotated, List, Optional
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
from huggingface_hub import login
import chromadb
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HUGGING_FACE_API_KEY")
model_ids = {
    "MPNet_doc_sql": "sentence-transformers/all-mpnet-base-v2"
    ,"baai_code":"BAAI/bge-base-en-v1.5"
}

cwd = os.getcwd()
db_path = os.path.abspath(os.path.join(cwd, "..", "chroma_db"))
client_chromadb = chromadb.PersistentClient(path=db_path)

In [ ]:
import requests
import numpy as np
from rank_bm25 import BM25Okapi 
import logging
import json

# Set up structured logging - forces everything onto clean single lines
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s'
)
logger = logging.getLogger("agent_debug")

In [55]:
#Embedding
def get_api_embeddings(texts, model_id):
    api_url = f"https://router.huggingface.co/hf-inference/models/{model_id}/pipeline/feature-extraction"

    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    
    payload = {
        "inputs": texts,
        "options": {"wait_for_model": True} 
    }
    
    response = requests.post(api_url, headers=headers, json=payload)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        return None
    
    result = response.json()
    arr= np.array(result)
    if arr.ndim == 3:  # mean pool token embeddings
        arr = arr.mean(axis=1)
    return arr 

In [56]:
# A local dictionary reducer
def merge_sources(left: dict, right: dict) -> dict:
    merged = left.copy() if left else {}
    for category, new_results in (right or {}).items():
        if category in merged and isinstance(merged[category], list):
            merged[category] = merged[category] + new_results
        else:
            merged[category] = new_results
    return merged

In [79]:
class AgentState(TypedDict):
    query: str              # "user query"
    mode: str               # "qa" or "impact"
    sub_mode: list[str]     # "code_base" , "doc" , "schema" - for "QA"
    retrieved_context: str  # "docs/code chunks retrieved from ChromaDB"
    answer: str             # llm answer
    sources: Annotated[dict, merge_sources]  # "Sources retrieved from chromadb" &  merge_sources : "Merges parallel dictionaries cleanly" 
    sources_cited : list[str] #Explicit list tracking for citations
    changed_functions: list[str] # extracted from PR diff
    impacted_files: list[str]   # files that call changed functions
    high_confidence_callers: list[dict]
    uncertain_candidates: list[dict]
    user_approval: Optional[str]

def empty_state(query: str) -> AgentState:
    """Always initialise ALL keys - prevents KeyError in any node."""
    return {
        "query": query, "mode": "","sub_mode": [], "retrieved_context": "",
        "answer": "", "sources": {}, "sources_cited":[],  "changed_functions": [],
        "impacted_files": [], "high_confidence_callers": [],
        "uncertain_candidates": [], "user_approval": None,
    }

In [80]:
def get_collection_safely(client, name):
    existing_collections = [c.name for c in client.list_collections()]
    
    if name not in existing_collections:
        raise ValueError(f"Collection '{name}' does not exist.")
        
    return client.get_collection(name)

# Usage
try:
    collection_doc = get_collection_safely(client_chromadb, "docbase_devcontext")
    collection_schema = get_collection_safely(client_chromadb, "schemabase_devcontext")
    collection_code = get_collection_safely(client_chromadb, "codebase_devcontext")
except ValueError as e:
    print(e)

In [81]:
#entry node
def route_intent(state: AgentState) -> dict :
    prompt = f"""
    Classify below query into qa or pr_impact .
    qa=asking about how code/system works or onboarding questions
    pr_impact=asking about what code breaks if code changes are done , pr review questions.
    Query = {state['query']}
    Respond with one word only : qa or pr_impact.
    """
    response = llm.invoke(prompt , max_tokens=5)
    mode=response.content.strip().lower()
    if mode not in ('qa','pr_impact'):
        mode='qa' #fallback
    
    logger.info(json.dumps({
        "node": "route_intent",
        "event": "intent_classification",
        "query_preview": state["query"][:60],
        "classified_mode": mode
    }))
    return {"mode":mode}

#Mapping intent 
def intent_router(state: AgentState) -> str:
    logger.info(json.dumps({
        "edge": "intent_router",
        "mode": state["mode"]
    }))
    return state["mode"]  # "qa" -> qa branch, "pr_impact" ->  pr branch 


In [82]:
def get_bm25 (collection) :
    all_data_doc = collection.get()
    documents = all_data_doc["documents"]
    metadatas = all_data_doc["metadatas"]
    ids = all_data_doc["ids"]

    tokenized = [re.findall(r"\b\w+\b", doc.lower()) for doc in documents]
    bm25 =  BM25Okapi(tokenized)

    mapping = {
        "documents":documents , "metadatas": metadatas , "ids" : ids
    }
    return bm25 , mapping

In [83]:
bm25_code , mapping_code = get_bm25(collection_code)
bm25_doc , mapping_doc = get_bm25(collection_doc)
bm25_schema , mapping_schema = get_bm25(collection_schema)

In [84]:
def bm25_search (bm_25 , query : str , mapping : dict , top_n: int = 3) -> list :
    tokenized_query = re.findall(r"\b\w+\b", query.lower())
    scores = bm_25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]
    bm25_results = []
    for idx in top_indices:
        bm25_results.append({
            "id": mapping["ids"][idx],
            "text": mapping["documents"][idx],
            "metadata": mapping["metadatas"][idx] or {}
        })

    return bm25_results

def semantic_search (collection : chromadb.Collection , embedding_model :str, query : str , top_n: int = 3) -> list :
    query_vec_embedding = get_api_embeddings(query,embedding_model)
    res = collection.query(query_embeddings=[query_vec_embedding], n_results=top_n)
    semantic_results=[]
    if res["documents"] :
        for i in range(len(res["documents"][0])):
            semantic_results.append({
                "id": res["ids"][0][i],
                "text": res["documents"][0][i],
                "metadata": res["metadatas"][0][i] or {}
            })
    
    return semantic_results

def rrf_fusion(bm25_results: list, semantic_results: list, k: int = 60, top_n: int = 3):
    fused_scores = {}
    
    def add_ranks(results):
        for rank, doc in enumerate(results):
            # Use the document text or a unique ID as the key
            doc_id = doc["id"]
            if doc_id not in fused_scores:
                fused_scores[doc_id] = {"doc": doc, "score": 0.0}
            # RRF Formula: 1 / (k + rank)
            fused_scores[doc_id]["score"] += 1.0 / (k + rank + 1)

    add_ranks(bm25_results)
    add_ranks(semantic_results)
    
    reranked = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in reranked[:top_n]]

In [85]:
import json

def llm_parallel_router_node(state: AgentState) -> dict:
    """
    Uses the LLM to select MULTIPLE target collections to search simultaneously.
    """
    prompt = f"""
    You are an expert search routing agent for a software repository RAG system.
    Analyze the user query and determine which collections must be searched to fully answer it. 
    You can select one, two, or all three options.
    
    Categories to choose from:
    - "codebase": For questions about syntax, function implementations, classes, or code logic.
    - "schema": For questions about database tables, columns, constraints, or SQL schemas.
    - "doc": For questions about architecture, high-level features, setup instructions, or how-to flows.
    
    Query: "{state['query']}"
    
    Respond ONLY with a JSON object containing a "targets" key mapped to a list of selected strings.
    Do not include markdown blocks, backticks, or conversational filler.
    
    Example response for an exploratory question: 
    {{"targets": ["doc", "codebase", "schema"]}}
    """
    response = llm.invoke(prompt, max_tokens=30)
    clean_content = response.content.strip()

    try:
        targets = json.loads(clean_content).get("targets", ["doc"])
    except Exception:
        targets = []
        if "code" in clean_content.lower(): targets.append("codebase")
        if "schema" in clean_content.lower(): targets.append("schema")
        if "doc" in clean_content.lower(): targets.append("doc")
        if not targets: targets = ["doc"] # Default safe fallback

    logger.info(json.dumps({
        "node": "llm_parallel_router_node",
        "event": "parallel_target_selection",
        "selected_sub_modes": targets
    }))    
    return {"sub_mode": targets} 

def get_search_nodes(state: AgentState) -> list[str]:
    # Since state["sub_mode"] is a list, e.g., ["codebase", "schema"]
    # mapping those strings directly to actual graph node names.
    nodes = []
    if "codebase" in state["sub_mode"]:
        nodes.append("search_codebase")
    if "schema" in state["sub_mode"]:
        nodes.append("search_schema")
    if "doc" in state["sub_mode"]:
        nodes.append("search_docs")

    logger.info(json.dumps({
        "edge": "get_search_nodes",
        "event": "fan_out_trigger",
        "active_parallel_nodes": nodes
    }))    
    # Safe fallback if the list is empty
    return nodes if nodes else ["search_docs"]

In [86]:
#QA node
# Helper to make database matches safe and compressed for JSON log dumps
def format_log_chunks(rrf_results: list) -> list:
    return [
        {
            "id": r.get("id", "unknown"),
            "text_snippet": r.get("text", "")[:80] + "...",
            "file": (r.get("metadata") or {}).get("file_name", "unknown")
        }
        for r in rrf_results
    ]

def search_codebase(state : AgentState) -> dict:
    """Search the codebase index for functions, classes, and callers. Use for code-related questions."""
    bm25_res=bm25_search(bm25_code , state["query"] , mapping_code )
    semantic_res=semantic_search(collection_code , model_ids["baai_code"] , state["query"] )
    rrf = rrf_fusion(bm25_res , semantic_res )

    logger.info(json.dumps({
        "node": "search_codebase",
        "event": "database_retrieval",
        "chunks_found": len(rrf),
        "matches": format_log_chunks(rrf)
    }))

    return {"sources": {"codebase": rrf}}

def search_docs(state : AgentState) -> dict:
    """Search internal documentation and README files. Use for how-to and architecture questions."""
    bm25_res=bm25_search(bm25_doc , state["query"] , mapping_doc )
    semantic_res=semantic_search(collection_doc , model_ids["MPNet_doc_sql"] , state["query"] )
    rrf = rrf_fusion(bm25_res , semantic_res )

    logger.info(json.dumps({
        "node": "search_docs",
        "event": "database_retrieval",
        "chunks_found": len(rrf),
        "matches": format_log_chunks(rrf)
    }))

    return {"sources": {"doc": rrf}} 


def search_schema(state: AgentState) -> dict:
    """Search schema for tables , columns , Sps , triggers etc. Use for data base related questions"""
    bm25_res=bm25_search(bm25_schema , state["query"] , mapping_schema )
    semantic_res=semantic_search(collection_schema , model_ids["MPNet_doc_sql"] , state["query"] )
    rrf = rrf_fusion(bm25_res , semantic_res )

    logger.info(json.dumps({
        "node": "search_schema",
        "event": "database_retrieval",
        "chunks_found": len(rrf),
        "matches": format_log_chunks(rrf)
    }))
    return {"sources": {"schema": rrf}}  


def aggregate_search_res(state: AgentState) -> dict:
    """
    Gathers contexts collected simultaneously across whichever categories executed.
    """
    context_chunks = []
    sources = state.get("sources", {})
    
    if "codebase" in sources and sources["codebase"]:
        code_str = "\n".join([s["text"] for s in sources["codebase"]])
        context_chunks.append(f"=== RELEVANT CODE SNIPPETS ===\n{code_str}")
        
    if "schema" in sources and sources["schema"]:
        schema_str = "\n".join([s["text"] for s in sources["schema"]])
        context_chunks.append(f"=== DATABASE SCHEMAS ===\n{schema_str}")
        
    if "doc" in sources and sources["doc"]:
        doc_str = "\n".join([s["text"] for s in sources["doc"]])
        context_chunks.append(f"=== DOCUMENTATION ===\n{doc_str}")

    logger.info(json.dumps({
        "node": "aggregate_search_res",
        "event": "context_consolidation",
        "total_payload_characters": len(context_chunks),
        "keys_merged": list(sources.keys())
    }))
    return {"retrieved_context": "\n\n".join(context_chunks)}

#QA answer generate 
def qa_retrieve_generate(state: AgentState) -> dict:

    prompt = (
        "You are a codebase assistant. Answer using ONLY the context below.\n"
        "Cite source keys used (e.g. codebase:authenticate_user, docs:auth-flow.md).\n\n"
        f"Context : {state["retrieved_context"]}\n\n"
        f"Query: {state['query']}\n\n"
        "Output format (two lines only):\n"
        "Sources: comma-separated source keys\n"
        "Answer: your answer"
    )
    response = llm.invoke(prompt)
    content = response.content.strip()

    sources, answer = [], content
    for line in content.splitlines():
        if line.lower().startswith("sources:"):
            sources = [s.strip() for s in line.split(":", 1)[1].split(",")]
        elif line.lower().startswith("answer:"):
            answer = line.split(":", 1)[1].strip()

    logger.info(json.dumps({
        "node": "qa_retrieve_generate",
        "event": "token_generation_complete",
        "sources_cited": sources
    }))
    
    return {
        "answer": answer
        ,"sources_cited" : sources
    }

In [87]:

#pr impact code.
def diff_parser (state : AgentState) -> dict:
    pattern = r'^[+-][\t ]*def\s+(\w+)\s*\('
    matches=re.findall(pattern , state['query'] , re.MULTILINE)

    logger.info(json.dumps({
        "node": "diff_parser",
        "event": "regex_diff_extraction",
        "functions_extracted": list(set(matches))
    }))

    return {"changed_functions" :list(set(matches))}

# Adjust this threshold based on how aggressively you want to trigger interrupts
CONFIDENCE_THRESHOLD = 0.025  

def dependency_finder(state: AgentState) -> dict:
    changed_fns = state.get("changed_functions", [])
    
    # Short-circuit if no functions were successfully parsed from the diff
    if not changed_fns:
        return {
            "retrieved_context": "No code changes detected in the PR query.",
            "impacted_files": [],
            "high_confidence_callers": [],
            "uncertain_candidates": [],
        }

    high_confidence = []
    uncertain = []
    context_chunks = []
    seen_chunks = set() # Prevent evaluating the exact same database chunk twice

    for fn_name in changed_fns:
        bm25_res = bm25_search(bm25_code, fn_name, mapping_code, top_n=5)
        semantic_res = semantic_search(collection_code, model_ids["baai_code"], fn_name, top_n=5)
        fused_matches = rrf_fusion(bm25_res, semantic_res, top_n=5)
        
        for match in fused_matches:
            chunk_id = match["id"]
            if chunk_id in seen_chunks:
                continue
            seen_chunks.add(chunk_id)
            
            meta = match["metadata"] or {}
            file_name = meta.get("file_name") or "unknown_file.py"
            matched_scope = "module_level" 
            clean_text = match["text"].replace(" ", "")
            is_explicit_call = f"{fn_name.strip()}(" in clean_text
            confidence_score = 0.05 if is_explicit_call else 0.01
            
            candidate = {
                "file": file_name,
                "function": f"{matched_scope} (ID: {chunk_id})",
                "score": confidence_score
            }
            
            context_chunks.append(f"--- File: {file_name} (Chunk ID: {chunk_id}) ---\n{match['text']}")
            if confidence_score >= CONFIDENCE_THRESHOLD:
                high_confidence.append(candidate)
            else:
                uncertain.append(candidate)

    logger.info(json.dumps({
        "node": "dependency_finder",
        "event": "dependency_analysis",
        "high_confidence_count": len(high_confidence),
        "uncertain_count": len(uncertain)
    }))

    return {
        "retrieved_context": "\n\n".join(context_chunks),
        "impacted_files": list(set([c["file"] for c in high_confidence])), # Unique file list
        "high_confidence_callers": high_confidence,
        "uncertain_candidates": uncertain,
    }

def find_callers(state: AgentState) -> dict:
    uncertain = state.get("uncertain_candidates", [])

    # Pass-through if nothing uncertain or already resolved
    if not uncertain or state.get("user_approval") is not None:
        return {}

    high_conf = state.get("high_confidence_callers", [])

    decision = interrupt({
        "message": "INTERRUPT - Low-confidence match(es) detected",
        "candidates": uncertain,
        "current_high_confidence": high_conf,
        "threshold": CONFIDENCE_THRESHOLD,
    })
    logger.info(json.dumps({
        "node": "find_callers",
        "event": "runtime_resume",
        "user_input_received": decision
    }))
    return {"user_approval": decision}

def impact_generate(state: AgentState) -> dict:
    final_list = list(state.get("high_confidence_callers", []))
    decision = state.get("user_approval", "n")

    if decision and decision.strip().lower() == "y":
        final_list.extend(state.get("uncertain_candidates", []))

    uncertain = state.get("uncertain_candidates", [])
    report_lines = ["\n=== Final PR Impact Report ==="]
    for c in final_list:
        report_lines.append(f"{c['file']}  -  {c['function']}  (score: {c['score']})")
    if decision and decision.strip().lower() == "n":
        for c in uncertain:
            report_lines.append(f"{c['file']}  -  dropped by engineer  (score: {c['score']})")
    report_lines.append("==============================")

    prompt = (
        "You are a code impact analyst. Explain what could break and why, "
        "grounded in the context and confirmed files only. Two sentences per file max.\n\n"
        f"Context:\n{state['retrieved_context']}\n\n"
        f"Confirmed impacted files: {[c['file'] for c in final_list]}"
    )
    response = llm.invoke(prompt)

    logger.info(json.dumps({
        "node": "impact_generate",
        "event": "pipeline_complete",
        "final_impacted_file_count": len(final_list)
    }))

    return {
        "impacted_files": [c["file"] for c in final_list],
        "answer": "\n".join(report_lines) + "\n\n" + response.content,
    }

In [88]:
memory = MemorySaver()
graph = StateGraph(AgentState)
graph.add_node('route_intent' , route_intent)

graph.add_node("llm_parallel_router_node", llm_parallel_router_node)
graph.add_node("search_codebase", search_codebase)
graph.add_node("search_docs", search_docs)
graph.add_node("search_schema", search_schema)
graph.add_node("aggregate_search_res", aggregate_search_res)
graph.add_node('qa_retrieve_generate' , qa_retrieve_generate)

graph.add_node('diff_parser' , diff_parser)
graph.add_node('dependency_finder' , dependency_finder)
graph.add_node("find_callers", find_callers) 
graph.add_node('impact_generate' , impact_generate)


graph.add_edge(START, 'route_intent')
graph.add_conditional_edges('route_intent' , intent_router , 
{
        "qa": "llm_parallel_router_node", 
        "pr_impact": "diff_parser"
})
graph.add_conditional_edges("llm_parallel_router_node",
    get_search_nodes,
    {
        "search_codebase": "search_codebase",
        "search_schema": "search_schema",
        "search_docs": "search_docs"
    }
)
graph.add_edge('search_codebase' , 'aggregate_search_res')
graph.add_edge('search_schema' , 'aggregate_search_res')
graph.add_edge('search_docs' , 'aggregate_search_res')
graph.add_edge('aggregate_search_res' , 'qa_retrieve_generate')
graph.add_edge('qa_retrieve_generate' , END)

graph.add_edge('diff_parser' , 'dependency_finder')
graph.add_edge('dependency_finder' , 'find_callers')
graph.add_edge('find_callers' , 'impact_generate')
graph.add_edge('impact_generate' , END)

app=graph.compile(checkpointer=memory)


In [89]:
# Visualize
print(app.get_graph().draw_ascii())

                                               +-----------+                                                
                                               | __start__ |                                                
                                               +-----------+                                                
                                                      *                                                     
                                                      *                                                     
                                                      *                                                     
                                              +--------------+                                              
                                           ...| route_intent |                                              
                                    .......   +--------------+                                              
                   

In [ ]:
ADVERSARIAL_CASES = [
    {
        "id": "A1",
        "label": "Missing file - not in index",
        "query": "What does the function `process_invoice_batch` do?",
        "expected_behavior": "Should say 'not found in index', not hallucinate a description",
        "expected_failure": "Agent may fabricate an answer from training weights"
    },
    {
        "id": "A2",
        "label": "Diff with no function signature changes",
        "query": "Analyze this PR diff for blast radius:\n--- a/config.yaml\n+++ b/config.yaml\n- timeout: 30\n+ timeout: 60",
        "expected_behavior": "Should say 'no code changes detected, config-only diff'",
        "expected_failure": "Agent may still run dependency analysis and return empty or confused result"
    },
    {
        "id": "A3",
        "label": "Ambiguous one-word query",
        "query": "auth",
        "expected_behavior": "Should ask a clarifying question - onboarding or impact analysis?",
        "expected_failure": "Agent picks a branch arbitrarily with no signal to user"
    },
    {
        "id": "A4",
        "label": "Cross-source confusion",
        "query": "What columns does the users table have and what service writes to it?",
        "expected_behavior": "Should call search_schema AND search_codebase, combine results",
        "expected_failure": "Agent may call only one tool and miss half the answer"
    },
    {
        "id": "A5",
        "label": "Repeated tool loop",
        "query": "Find every file that imports the DatabaseConnection class",
        "expected_behavior": "Should search once, return results or 'not found'",
        "expected_failure": "Agent may re-call search_codebase 3+ times with slightly different queries"
    }
]


In [91]:
import time

In [92]:
evaluation_results = {}

logger.info("Starting DevContext AI Stress-Testing Protocol...")

for case in [ADVERSARIAL_CASES[0]]:
    case_id = case["id"]
    logger.info(f"Executing Stress Case {case_id}: {case['label']}")
    state = empty_state(query=case["query"])
    state["sources"] = {}  
    
    print(f"USER QUERY:\n{case['query']}")
    config = {"configurable": {"thread_id": f"stress_thread_{case_id}_{int(time.time())}"}}
    
    try:
        app.invoke(state, config)
        
        final_state = app.get_state(config).values
        evaluation_results[case_id] = {
            "label": case["label"],
            "mode_chosen": final_state.get("mode"),
            "sub_modes_triggered": final_state.get("sub_mode"),
            "sources_found": list(final_state.get("sources", {}).keys()),
            "agent_response": final_state.get("answer")
        }
        agent_answer = final_state.get("answer", "No response text generated.")

    except Exception as e:
        if "interrupt" in str(e).lower() or "Command" in str(type(e)):
            logger.warning(f"Case {case_id} caught by human gate. Resuming with negative ('n') clearance.")
            
            app.invoke(Command(resume="n"), config)
            final_state = app.get_state(config).values
            
            evaluation_results[case_id] = {
                "label": case["label"],
                "mode_chosen": final_state.get("mode"),
                "sub_modes_triggered": final_state.get("sub_mode"),
                "sources_found": list(final_state.get("sources", {}).keys()),
                "agent_response": final_state.get("answer") + " (Halted & Rejected via Interrupt)"
            }
            agent_answer = final_state.get("answer", "") + "\n (Halted: Dropped via Interrupt Node Decision)"
        else:
            logger.error(f"Execution failed on case {case_id}: {str(e)}")
            evaluation_results[case_id] = {
                "label": case["label"],
                "mode_chosen": "CRASHED",
                "sub_modes_triggered": [],
                "sources_found": [],
                "agent_response": f"Runtime Error: {str(e)}"
            }
            agent_answer = f"PIPELINE CRASHED: {str(e)}"
            final_state = {}

    print(f"AGENT ANSWER:\n{agent_answer}")
    print("=" * 60)        
    # brief cooling period for llm for rpd/rpm restrictions
    time.sleep(2)

# --- POST-EVALUATION MATRIX DISPLAY ---
print("\n" + "="*80)
print("ADVERSARIAL EVALUATION REPORT SUMMARY MATRIX")
print("="*80)

for cid, metrics in evaluation_results.items():
    print(f"\n[Case {cid}]: {metrics['label']}")
    print(f"  ↳ Intent Route Mode  : {metrics['mode_chosen']}")
    print(f"  ↳ Sub-search Formats : {metrics['sub_modes_triggered']}")
    print(f"  ↳ Storage Keys Found : {metrics['sources_found']}")
    print(f"  ↳ Agent Final Answer : {metrics['agent_response']}")
    print("-" * 50)
    
print("\n" + "="*80)

2026-05-16 15:37:08,255 INFO Starting DevContext AI Stress-Testing Protocol...
2026-05-16 15:37:08,259 INFO Executing Stress Case A1: Missing file — not in index
2026-05-16 15:37:08,444 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 15:37:08,451 INFO {"node": "route_intent", "event": "intent_classification", "query_preview": "What does the function `process_invoice_batch` do?", "classified_mode": "qa"}
2026-05-16 15:37:08,455 INFO {"edge": "intent_router", "mode": "qa"}


USER QUERY:
What does the function `process_invoice_batch` do?


2026-05-16 15:37:09,734 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 15:37:09,745 INFO {"node": "llm_parallel_router_node", "event": "parallel_target_selection", "selected_sub_modes": ["codebase", "doc"]}
2026-05-16 15:37:09,748 INFO {"edge": "get_search_nodes", "event": "fan_out_trigger", "active_parallel_nodes": ["search_codebase", "search_docs"]}
2026-05-16 15:37:10,835 INFO {"node": "search_docs", "event": "database_retrieval", "chunks_found": 2, "matches": [{"id": "doc_0", "text_snippet": "## Auth System\nAuthentication uses JWT tokens. The `authenticate_user` function ...", "file": "onboarding_guide_md"}, {"id": "doc_1", "text_snippet": "## Payments\nAll payments route through Stripe. The `process_payment` function is...", "file": "onboarding_guide_md"}]}
2026-05-16 15:37:14,440 INFO {"node": "search_codebase", "event": "database_retrieval", "chunks_found": 3, "matches": [{"id": "code_index_1", "text_snippet": "def process_p

AGENT ANSWER:
There is no information about the function `process_invoice_batch` in the provided context.

ADVERSARIAL EVALUATION REPORT SUMMARY MATRIX

[Case A1]: Missing file — not in index
  ↳ Intent Route Mode  : qa
  ↳ Sub-search Formats : ['codebase', 'doc']
  ↳ Storage Keys Found : ['codebase', 'doc']
  ↳ Agent Final Answer : There is no information about the function `process_invoice_batch` in the provided context.
--------------------------------------------------



In [93]:
evaluation_results = {}

logger.info("Starting DevContext AI Stress-Testing Protocol...")

for case in ADVERSARIAL_CASES[1:]:
    case_id = case["id"]
    logger.info(f"Executing Stress Case {case_id}: {case['label']}")
    state = empty_state(query=case["query"])
    state["sources"] = {}  
    
    print(f"USER QUERY:\n{case['query']}")
    config = {"configurable": {"thread_id": f"stress_thread_{case_id}_{int(time.time())}"}}
    
    try:
        app.invoke(state, config)
        
        final_state = app.get_state(config).values
        evaluation_results[case_id] = {
            "label": case["label"],
            "mode_chosen": final_state.get("mode"),
            "sub_modes_triggered": final_state.get("sub_mode"),
            "sources_found": list(final_state.get("sources", {}).keys()),
            "agent_response": final_state.get("answer")
        }
        agent_answer = final_state.get("answer", "No response text generated.")

    except Exception as e:
        if "interrupt" in str(e).lower() or "Command" in str(type(e)):
            logger.warning(f"Case {case_id} caught by human gate. Resuming with negative ('n') clearance.")
            
            app.invoke(Command(resume="n"), config)
            final_state = app.get_state(config).values
            
            evaluation_results[case_id] = {
                "label": case["label"],
                "mode_chosen": final_state.get("mode"),
                "sub_modes_triggered": final_state.get("sub_mode"),
                "sources_found": list(final_state.get("sources", {}).keys()),
                "agent_response": final_state.get("answer") + " (Halted & Rejected via Interrupt)"
            }
            agent_answer = final_state.get("answer", "") + "\n (Halted: Dropped via Interrupt Node Decision)"
        else:
            logger.error(f"Execution failed on case {case_id}: {str(e)}")
            evaluation_results[case_id] = {
                "label": case["label"],
                "mode_chosen": "CRASHED",
                "sub_modes_triggered": [],
                "sources_found": [],
                "agent_response": f"Runtime Error: {str(e)}"
            }
            agent_answer = f"PIPELINE CRASHED: {str(e)}"
            final_state = {}

    print(f"AGENT ANSWER:\n{agent_answer}")
    print("=" * 60)        
    # brief cooling period for llm for rpd/rpm restrictions
    time.sleep(2)

# --- POST-EVALUATION MATRIX DISPLAY ---
print("\n" + "="*80)
print("ADVERSARIAL EVALUATION REPORT SUMMARY MATRIX")
print("="*80)

for cid, metrics in evaluation_results.items():
    print(f"\n[Case {cid}]: {metrics['label']}")
    print(f"  ↳ Intent Route Mode  : {metrics['mode_chosen']}")
    print(f"  ↳ Sub-search Formats : {metrics['sub_modes_triggered']}")
    print(f"  ↳ Storage Keys Found : {metrics['sources_found']}")
    print(f"  ↳ Agent Final Answer : {metrics['agent_response']}")
    print("-" * 50)
    
print("\n" + "="*80)

2026-05-16 15:38:28,852 INFO Starting DevContext AI Stress-Testing Protocol...
2026-05-16 15:38:28,857 INFO Executing Stress Case A2: Diff with no function signature changes


USER QUERY:
Analyze this PR diff for blast radius:
--- a/config.yaml
+++ b/config.yaml
- timeout: 30
+ timeout: 60


2026-05-16 15:38:29,197 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 15:38:29,204 INFO {"node": "route_intent", "event": "intent_classification", "query_preview": "Analyze this PR diff for blast radius:\n--- a/config.yaml\n+++", "classified_mode": "pr_impact"}
2026-05-16 15:38:29,209 INFO {"edge": "intent_router", "mode": "pr_impact"}
2026-05-16 15:38:29,221 INFO {"node": "diff_parser", "event": "regex_diff_extraction", "functions_extracted": []}
2026-05-16 15:38:29,673 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 15:38:29,687 INFO {"node": "impact_generate", "event": "pipeline_complete", "final_impacted_file_count": 0}


AGENT ANSWER:

=== Final PR Impact Report ===

Since there are no confirmed impacted files, there is no code to analyze for potential breaks. As a result, I have no findings to report, and no potential issues can be identified without further context or files to review.


2026-05-16 15:38:31,703 INFO Executing Stress Case A3: Ambiguous one-word query


...

AGENT ANSWER:
The users table has the following columns: id, username, password_hash, created_at, and the services that write to it include authenticate_user and potentially send_welcome_email (for fetching) but primarily user creation which is not shown.


2026-05-16 15:38:43,266 INFO Executing Stress Case A5: Repeated tool loop


USER QUERY:
Find every file that imports the DatabaseConnection class


2026-05-16 15:38:43,484 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 15:38:43,490 INFO {"node": "route_intent", "event": "intent_classification", "query_preview": "Find every file that imports the DatabaseConnection class", "classified_mode": "qa"}
2026-05-16 15:38:43,497 INFO {"edge": "intent_router", "mode": "qa"}
2026-05-16 15:38:43,731 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 15:38:43,734 INFO {"node": "llm_parallel_router_node", "event": "parallel_target_selection", "selected_sub_modes": ["codebase"]}
2026-05-16 15:38:43,735 INFO {"edge": "get_search_nodes", "event": "fan_out_trigger", "active_parallel_nodes": ["search_codebase"]}
2026-05-16 15:38:44,459 INFO {"node": "search_codebase", "event": "database_retrieval", "chunks_found": 3, "matches": [{"id": "code_index_1", "text_snippet": "def process_payment(order_id, amount, method):\n    \"\"\"Charges the given metho

AGENT ANSWER:
database.py, users.py, orders.py

ADVERSARIAL EVALUATION REPORT SUMMARY MATRIX

[Case A2]: Diff with no function signature changes
  ↳ Intent Route Mode  : pr_impact
  ↳ Sub-search Formats : []
  ↳ Storage Keys Found : []
  ↳ Agent Final Answer : 
=== Final PR Impact Report ===

Since there are no confirmed impacted files, there is no code to analyze for potential breaks. As a result, I have no findings to report, and no potential issues can be identified without further context or files to review.
--------------------------------------------------

[Case A3]: Ambiguous one-word query
  ↳ Intent Route Mode  : qa
  ↳ Sub-search Formats : ['codebase', 'doc', 'schema']
  ↳ Storage Keys Found : ['codebase', 'doc', 'schema']
  ↳ Agent Final Answer : The auth system uses JWT tokens, which expire after 24 hours, and are generated by the authenticate_user function.
--------------------------------------------------

[Case A4]: Cross-source confusion
  ↳ Intent Route Mode  : qa
  